# Exercise 8 - Ensemble on MNIST data

In [13]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier

In [2]:
mnist = fetch_openml("mnist_784", version=1, as_frame=False)
X, y = mnist.data, mnist.target.astype(np.uint8)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=10000, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=10000, random_state=42, stratify=y_train)
print(X_train.shape, X_val.shape, X_test.shape)

(50000, 784) (10000, 784) (10000, 784)


In [4]:
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
extra = ExtraTreesClassifier(n_estimators=100, random_state=42, n_jobs=-1)
svm = SVC(kernel="rbf", probability=True, random_state=42)

rf.fit(X_train, y_train)
extra.fit(X_train, y_train)
svm.fit(X_train, y_train)
models = {"Random Forest": rf, "Extra Trees": extra, "SVM": svm}
for name, model in models.items():
    print(name, accuracy_score(y_val, model.predict(X_val)))

Random Forest 0.9709
Extra Trees 0.9731
SVM 0.9798


In [6]:
voting = VotingClassifier(estimators=[("rf", rf), ("extra", extra), ("svm", svm)], voting="hard", n_jobs=-1)
voting.fit(X_train, y_train)
y_val_pred = voting.predict(X_val)

print("Voting accuracy:", accuracy_score(y_val, y_val_pred))

Voting accuracy: 0.9752


In [7]:
for name, model in models.items():
    print(name, accuracy_score(y_val, model.predict(X_val)))
print("Voting", accuracy_score(y_val, y_val_pred))

Random Forest 0.9709
Extra Trees 0.9731
SVM 0.9798
Voting 0.9752


In [8]:
soft_voting = VotingClassifier(estimators=[("rf", rf), ("extra", extra), ("svm", svm)], voting="soft", n_jobs=-1)
soft_voting.fit(X_train, y_train)
y_val_pred_soft = soft_voting.predict(X_val)

print("Soft voting accuracy:", accuracy_score(y_val, y_val_pred_soft))

Soft voting accuracy: 0.981


In [9]:
best_ensemble = voting
y_test_pred = best_ensemble.predict(X_test)
ensemble_acc = accuracy_score(y_test, y_test_pred)
print("Ensemble test accuracy:", ensemble_acc)

for name, model in models.items():
    acc = accuracy_score(y_test, model.predict(X_test))
    print(name, acc)

Ensemble test accuracy: 0.9728
Random Forest 0.9657
Extra Trees 0.9706
SVM 0.9787


In [10]:
for name, model in models.items():
    acc = accuracy_score(y_test, model.predict(X_test))
    print(f"{name}: {acc:.4f}")
    print(f"Improvement: {(ensemble_acc - acc) * 100:.2f} percentage points")

Random Forest: 0.9657
Improvement: 0.71 percentage points
Extra Trees: 0.9706
Improvement: 0.22 percentage points
SVM: 0.9787
Improvement: -0.59 percentage points


The important idea here is isn't simply that "three models are better than one." It's that different classifiers make different mistakes. Combining their predictions can cancel out some individual errors which is the basic motivation behind voting ensembles.

# Exercise 9 - Blending and Stacking on MNIST data

In [16]:
val_predictions = np.column_stack([rf.predict(X_val), extra.predict(X_val), svm.predict(X_val)])
print(val_predictions.shape)

(10000, 3)


In [17]:
blender = LogisticRegression(max_iter=1000)
blender.fit(val_predictions, y_val)

test_predictions = np.column_stack([rf.predict(X_test), extra.predict(X_test), svm.predict(X_test)])
y_test_stacking = blender.predict(test_predictions)

stacking_acc = accuracy_score(y_test, y_test_stacking)
print("Manual stacking accuracy:", stacking_acc)

print(f"Random Forest: {accuracy_score(y_test, rf.predict(X_test)):.4f}")
print(f"Extra Trees:   {accuracy_score(y_test, extra.predict(X_test)):.4f}")
print(f"SVM:           {accuracy_score(y_test, svm.predict(X_test)):.4f}")
print(f"Hard voting:   {ensemble_acc:.4f}")
print(f"Soft voting:   {accuracy_score(y_val, y_val_pred_soft):.4f} validation")
print(f"Stacking:      {stacking_acc:.4f}")

C:\Users\sumuk\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Manual stacking accuracy: 0.9659
Random Forest: 0.9657
Extra Trees:   0.9706
SVM:           0.9787
Hard voting:   0.9728
Soft voting:   0.9810 validation
Stacking:      0.9659


In [18]:
soft_test_acc = accuracy_score(y_test, soft_voting.predict(X_test))
print("Soft voting test:", soft_test_acc)
print("Stacking test:", stacking_acc)

Soft voting test: 0.9793
Stacking test: 0.9659


In [19]:
stacking = StackingClassifier(estimators=[("rf", rf), ("extra", extra), ("svm", svm)], final_estimator=LogisticRegression(max_iter=1000),
    cv="prefit", stack_method="predict", n_jobs=-1)
stacking.fit(X_val, y_val)

y_test_stack = stacking.predict(X_test)
stacking_acc = accuracy_score(y_test, y_test_stack)
print("StackingClassifier test accuracy:", stacking_acc)

C:\Users\sumuk\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


StackingClassifier test accuracy: 0.9659


For this data split, soft voting performs best on the test set followed by SVM.